# 39 — Scaling to n=4: quads on the bf16 baseline

Extends the nested scaling suite one level: **pair ⊂ triple ⊂ quad**. Each
of notebook 33's 30 triples is extended with a fourth pool concept chosen
by the same entanglement rule that built the triple, matched to the
triple's own `add_regime` (distant triples extended distantly, etc.) —
so every quad has its n=2 and n=3 ancestors already measured (33 for the
plain families, 34 for common-blend), and only n=4 runs here.

**Arms (30 quads each):**
- **blend / vote / schedule** — continuity: completes each method's
  n = 2 → 3 → 4 curve. Watch: schedule's per-concept budget falls to 6
  steps; blend's >80% all-improved (cosine).
- **common-blend** — the localization probe: does common-only ≈ blend
  survive as four concepts share a thinner common subspace?
- **schedule-scaled** — schedule at `steps = 32`, restoring the 8 active
  steps per concept it had at n=3: separates capacity contention from
  composition difficulty. Evaluated against a length-matched baseline;
  both metric tracks are per-position means, so lengths compare fairly.

Caveats carried from the suite: `min` over 4 deltas shrinks
stochastically with n — compare within-n rankings and per-method drops,
not raw joints across n.


In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch

from frames.evaluation.harness import EvaluationHarness
from frames.linalg.gds import common_blend
from frames.representations import FrameUnembeddingRepresentation
from frames.representations.concept import Concept
from frames.representations.schedulers import RoundRobin

In [ ]:
MIN_LEMMAS = 11
MAX_TOKENS = 3
K = 4
STEPS = 24
STEPS_SCALED = 32  # 8 active steps per concept at n=4, matching n=3's budget
COMMON_THRESHOLD = 1.05

MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"
MODEL_KEY = "llama-3.1-8b-bf16"

POOL = json.loads(Path("resources/32_concept_pool_bf16.json").read_text())
POOL_NAMES = [r["synset"] for r in POOL["pool"]]
GROUPS33 = json.loads(Path("resources/33_scaling_groups_bf16.json").read_text())
TRIPLES = GROUPS33["triples"]


def rho(a: str, b: str) -> float:
    key = f"{a}|{b}" if f"{a}|{b}" in POOL["pairwise_rho"] else f"{b}|{a}"
    return POOL["pairwise_rho"][key]


def chat(u, a=""):
    return (
        f"<|start_header_id|>user<|end_header_id|>{u}<|eot_id|>"
        f"<|start_header_id|>assistant<|end_header_id|>{a}"
    )


QUESTIONS = [
    ("What men can be?", "1."),
    ("What women can be?", "1."),
    ("Tell me a short story.", "Once"),
    ("Describe your favorite place.", "My favorite place is"),
    ("What do you like to do on weekends?", "I like"),
    ("Give me some advice for a good life.", "1."),
    ("What is the most important thing in the world?", "The most important thing is"),
    ("Describe the weather today.", "Today the weather is"),
    ("What should I cook for dinner?", "You should"),
    ("Name things that make people happy.", "1."),
]

LOG_PATH = Path("resources/39_scaling4.jsonl")
QUADS_PATH = Path("resources/39_quads.json")
LOG_PATH.unlink(missing_ok=True)

# extend each triple with a 4th concept at the triple's own add_regime
QUADS = []
for t in TRIPLES:
    members = [t["a"], t["b"], t["c"]]
    ranked = sorted(
        (x for x in POOL_NAMES if x not in members),
        key=lambda x: max(rho(x, m) for m in members),
    )
    pick = {
        "distant": ranked[0],
        "mid": ranked[len(ranked) // 2],
        "close": ranked[-1],
    }[t["add_regime"]]
    quad = [*members, pick]
    QUADS.append(
        {
            "a": t["a"],
            "b": t["b"],
            "c": t["c"],
            "d": pick,
            "add_regime": t["add_regime"],
            "pair_rho": t["pair_rho"],
            "mean_rho": sum(
                rho(x, y) for i, x in enumerate(quad) for y in quad[i + 1 :]
            )
            / 6,
        }
    )
QUADS_PATH.write_text(json.dumps({"quads": QUADS}, indent=1))
print(f"{len(QUADS)} quads -> {QUADS_PATH}")

In [ ]:
fur = FrameUnembeddingRepresentation.from_model_id(
    device_map="auto", id=MODEL_ID, torch_dtype=torch.bfloat16
)
prompts = [chat(u, a) for u, a in QUESTIONS]
harness = EvaluationHarness(
    fur, LOG_PATH, min_lemmas_per_synset=MIN_LEMMAS, max_token_count=MAX_TOKENS
)
baseline_texts = harness.generate_baseline(prompts, max_new_tokens=STEPS + 2)
baseline_scaled = harness.generate_baseline(prompts, max_new_tokens=STEPS_SCALED + 2)

used = sorted({s for q in QUADS for s in (q["a"], q["b"], q["c"], q["d"])})
concept = {}
for name in used:
    c = fur.get_concept_cached(name, MIN_LEMMAS, MAX_TOKENS)
    concept[name] = Concept(synset=c.synset, tensor=c.tensor.to(fur.model.device))
print(f"{len(concept)} concepts built")

skipped = []
for q in QUADS:
    synsets = [q["a"], q["b"], q["c"], q["d"]]
    cs = [concept[s] for s in synsets]
    stack = torch.cat(Concept._pad_to_common_rank(cs))
    try:
        cb_guide = Concept(
            synset=" + ".join(synsets) + " (common)",
            tensor=common_blend(stack, common_threshold=COMMON_THRESHOLD).unsqueeze(0),
        )
    except ValueError:
        cb_guide = None
        skipped.append(synsets)
    arms = {
        "blend": (
            STEPS,
            baseline_texts,
            lambda: fur.generate_with_topk_guide(
                prompts, guide=Concept.average(cs), k=K, steps=STEPS
            ),
        ),
        "vote": (
            STEPS,
            baseline_texts,
            lambda: fur.generate_with_topk_multi_guide(
                prompts, guides=cs, k=K, steps=STEPS, normalize="zscore"
            ),
        ),
        "schedule": (
            STEPS,
            baseline_texts,
            lambda: fur.generate_with_topk_multi_guide(
                prompts, guides=cs, k=K, steps=STEPS, scorer=RoundRobin()
            ),
        ),
        "schedule-scaled": (
            STEPS_SCALED,
            baseline_scaled,
            lambda: fur.generate_with_topk_multi_guide(
                prompts, guides=cs, k=K, steps=STEPS_SCALED, scorer=RoundRobin()
            ),
        ),
    }
    if cb_guide is not None:
        arms["common-blend"] = (
            STEPS,
            baseline_texts,
            lambda: fur.generate_with_topk_guide(
                prompts, guide=cb_guide, k=K, steps=STEPS
            ),
        )
    for method, (steps, base_texts, generate) in arms.items():
        torch.cuda.empty_cache()
        texts, probe = generate()
        harness.evaluate(
            prompts,
            texts,
            synsets,
            config={
                "model": MODEL_KEY,
                "method": method,
                "n": 4,
                "steps": steps,
                **q,
            },
            probe=probe,
            baseline_texts=base_texts,
            concepts=dict(zip(synsets, cs)),
            angles=True,
        )
    print(f"{' + '.join(s.split('.')[0] for s in synsets)} ({q['add_regime']}): done")

print(f"\n{len(skipped)} quads skipped for common-blend (no common subspace)")

In [ ]:
def rows_from(path, methods):
    rows = []
    for line in Path(path).read_text().strip().split("\n"):
        r = json.loads(line)
        c = r["config"]
        if c["method"] not in methods:
            continue
        synsets = [s for s in (c["a"], c["b"], c.get("c"), c.get("d")) if s]
        deltas = [r["expression"][s]["delta"] for s in synsets]
        cosines = [r["angles"][s]["delta"]["token_cos"] for s in synsets]
        rows.append(
            {
                "method": c["method"],
                "n": c["n"],
                "joint": min(deltas),
                "joint_cos": min(cosines),
                "all_positive": min(deltas) > 0,
                "all_positive_cos": min(cosines) > 0,
                "add_regime": c.get("add_regime"),
                "ppl_ratio": r["ppl_ratio"],
            }
        )
    return pd.DataFrame(rows)


fresh = rows_from(
    LOG_PATH, {"blend", "vote", "schedule", "schedule-scaled", "common-blend"}
)
hist33 = rows_from("resources/33_scaling3_bf16.jsonl", {"blend", "vote", "schedule"})
hist34 = rows_from("resources/34_common_blend.jsonl", {"common-blend"})
results = pd.concat([fresh, hist33, hist34], ignore_index=True)
print(f"gate: {results.groupby(['n']).size().to_dict()} records")

summary = (
    results.groupby(["method", "n"])
    .agg(
        joint=("joint", "mean"),
        joint_cos=("joint_cos", "mean"),
        all_positive=("all_positive", "mean"),
        all_positive_cos=("all_positive_cos", "mean"),
        ppl_ratio=("ppl_ratio", "mean"),
        n_runs=("joint", "size"),
    )
    .reset_index()
)
summary.to_csv("resources/39_scaling4_summary.csv", index=False)
summary.round(4)

In [ ]:
methods = ["blend", "common-blend", "vote", "schedule", "schedule-scaled"]
colors = {
    "blend": "tab:blue",
    "common-blend": "tab:red",
    "vote": "tab:orange",
    "schedule": "tab:green",
    "schedule-scaled": "darkgreen",
}
fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))
for ax, (metric, label) in zip(
    axes,
    (
        ("joint_cos", "mean joint token-cosine gain"),
        ("all_positive_cos", "share ALL concepts improved (cosine)"),
    ),
):
    for method in methods:
        sub = summary[summary["method"] == method].sort_values("n")
        ax.plot(
            sub["n"],
            sub[metric],
            marker="o",
            color=colors[method],
            label=method,
            ls="--" if method == "schedule-scaled" else "-",
        )
    ax.set_xticks([2, 3, 4])
    ax.set_xlabel("concurrent concepts n")
    ax.set_ylabel(label)
    ax.axhline(0, color="gray", lw=0.8)
axes[1].axhline(0.5, color="gray", ls=":", lw=0.8)
axes[0].legend(fontsize=8)
plt.suptitle("Scaling curves n=2..4 (bf16 pool, nested groups)")
plt.tight_layout()
plt.savefig("resources/39_scaling_curves.png", dpi=150)
plt.show()

print("per-method drop n=3 -> n=4 (joint_cos):")
for method in methods:
    sub = summary[summary["method"] == method].set_index("n")
    if 3 in sub.index and 4 in sub.index:
        print(
            f"  {method:16s} {sub.loc[3, 'joint_cos']:.4f} -> {sub.loc[4, 'joint_cos']:.4f}"
        )
n4 = summary[summary["n"] == 4].set_index("method")
if "schedule-scaled" in n4.index:
    print(
        f"\ncapacity effect at n=4: schedule {n4.loc['schedule', 'joint_cos']:.4f}"
        f" vs schedule-scaled {n4.loc['schedule-scaled', 'joint_cos']:.4f}"
    )
if "common-blend" in n4.index:
    print(
        f"localization at n=4: blend {n4.loc['blend', 'joint_cos']:.4f}"
        f" vs common-blend {n4.loc['common-blend', 'joint_cos']:.4f}"
    )

In [ ]:
def answer(text: str) -> str:
    marker = "assistant<|end_header_id|>"
    return text.split(marker)[-1].strip() if marker in text else text


shown = 0
for line in Path(LOG_PATH).read_text().strip().split("\n"):
    r = json.loads(line)
    if r["config"]["method"] != "blend" or "short story" not in r["prompt"]:
        continue
    deltas = {k.split(".")[0]: round(v["delta"], 2) for k, v in r["expression"].items()}
    print(f"[blend n=4 / {r['config']['add_regime']}] {deltas}")
    print("   ", answer(r["text"])[:170], "\n")
    shown += 1
    if shown == 3:
        break